In [89]:
import json
import os
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast


In [90]:
results_df = pd.read_csv("../data/output/BAttn_lstm2_result_t70_10.csv")
results_df['evidences_id'] = results_df['evidences_id'].apply(ast.literal_eval)
results_df

,claim_id,evidences_id
0,claim-752,"[evidence-215, evidence-298, evidence-315, evi..."
1,claim-375,"[evidence-215, evidence-298, evidence-315, evi..."
2,claim-1266,"[evidence-215, evidence-298, evidence-315, evi..."
3,claim-871,"[evidence-215, evidence-298, evidence-315, evi..."
4,claim-2164,"[evidence-215, evidence-298, evidence-315, evi..."
...,...,...
149,claim-2400,"[evidence-215, evidence-298, evidence-315, evi..."
150,claim-204,"[evidence-215, evidence-298, evidence-315, evi..."
151,claim-1426,"[evidence-215, evidence-298, evidence-315, evi..."
152,claim-698,"[evidence-215, evidence-298, evidence-315, evi..."


In [138]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	filtered_tokens = [stemmer.stem(word) for word in filtered_tokens if not word.isnumeric()]
	return ' '.join(filtered_tokens)

def preprocess_text_keep_stopwords(text):
	"""Preprocesses text by lowercasing, tokenizing, stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum()]
	return ' '.join(filtered_tokens)

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [176]:
dev_claims_data = load_data('../data/dev-claims.json')

data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_cls': preprocess_text_keep_stopwords(claim_details['claim_text'])
        })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)
dev_claims_df

,claim_id,claim,claim_text_cls
0,claim-752,south australia expen electr world,south australia ha the most expens electr in t...
1,claim-375,per cent total annual global emiss carbon diox...,when 3 per cent of total annual global emiss o...
2,claim-1266,mean world 1c warmer time,thi mean that the world is now 1c warmer than ...
3,claim-871,happen zika may also good model second worri e...,as it happen zika may also be a good model of ...
4,claim-2164,greenland lost tini fraction ice mass,greenland ha onli lost a tini fraction of it i...
...,...,...,...
149,claim-2400,suddenli label co2 pollut disserv ga play enor...,suddenli label co2 as a pollut is a disservic ...
150,claim-204,natur orbit driven warm atmosph carbon dioxid ...,after a natur orbit driven warm atmospher carb...
151,claim-1426,mani world coral reef alreadi barren state con...,mani of the world s coral reef are alreadi bar...
152,claim-698,recent studi led lawrenc livermor nation labor...,a recent studi led by lawrenc livermor nation ...


In [93]:
all_evidence_id = set([eid for evidences in results_df['evidences_id'].tolist() for eid in evidences])
len(all_evidence_id)

9380

In [95]:
evidence_data = load_data('../data/curated/mild_nostopwords_filtered_evidence.json')
evidence_df = pd.DataFrame(evidence_data.items(), columns=['id', 'evidence'])
evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur and agric...
1,evidence-1,lindberg began his profession career at the ag...
2,evidence-3,gerald franci goyer born octob was a professio...
3,evidence-7,in addit to known and tangibl risk unforese bl...
4,evidence-12,he is the current aida individu world champion...
...,...,...
298683,evidence-895097,snowflak fell on 19 out of 28 day in the bosto...
298684,evidence-906284,the long australian millenni drought broke in ...
298685,evidence-1025757,in addit mani area are experienc higher than n...
298686,evidence-403673,global warm of 1.5


In [116]:
evidence_df = evidence_df.loc[evidence_df['id'].isin(all_evidence_id)]
evidence_df = evidence_df.reset_index(drop=True)
evidence_df

,id,evidence,evidence_tfidf
0,evidence-215,this allow earth surfac to be warm enough to h...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,evidence-298,a global warm conspiraci theori invok claim th...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,evidence-315,as el niño condit start to develop dure earli ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,evidence-320,concern about genet divers are therefor especi...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,evidence-337,the had it warmest on record in 2012,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...
9375,evidence-895097,snowflak fell on 19 out of 28 day in the bosto...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9376,evidence-906284,the long australian millenni drought broke in ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9377,evidence-1025757,in addit mani area are experienc higher than n...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9378,evidence-403673,global warm of 1.5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [98]:
all_claim_text = dev_claims_df['claim'].tolist()
all_evidence_text = evidence_df['evidence'].tolist()

In [177]:
# Vectorization
vectorizer = TfidfVectorizer(stop_words="english")

# Fit the vectorizer on both claims and evidences
claim_vec = vectorizer.fit_transform(all_claim_text)
dev_claims_df['claim_tfidf'] = list(claim_vec.toarray()) 

evidence_vec = vectorizer.transform(all_evidence_text)
evidence_df['evidence_tfidf'] = list(evidence_vec.toarray())

In [178]:
def top_k_evidence(claim_df, evidence_df, evidence_map, k=5, threshold = 0.5):
	# compute cosine similarity between each claim and each evidence
	X = np.array(claim_df['claim_tfidf'].values.tolist())
	y = np.array(evidence_df['evidence_tfidf'].values.tolist())
	sim = cosine_similarity(X, y)

	# get top k evidence with highest similarity score with the claim
	top_evidence_id = []
	for i in range(sim.shape[0]):
		data = np.argwhere(sim[i] > threshold)
		top_evidence_id.append([evidence_df.iloc[int(ind[0])]['id'] for ind in data])

	claim_df['top5_evidence_id'] = top_evidence_id

	claim_df = claim_df[["claim_id", "claim_text_cls", "top5_evidence_id"]]

	# get texts of top k evidence
	claim_df['evidence_texts'] = claim_df['top5_evidence_id'].apply(
		lambda x: [evidence_map[evidence_id] for evidence_id in x]
	)
	return claim_df

In [179]:
dev_claims_df = top_k_evidence(dev_claims_df, evidence_df, evidence_data)

/var/folders/df/4qk5nt6555bggnc39502n5b80000gn/T/ipykernel_19508/576050445.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  claim_df['evidence_texts'] = claim_df['top5_evidence_id'].apply(


In [180]:
dev_claims_df

,claim_id,claim_text_cls,top5_evidence_id,evidence_texts
0,claim-752,south australia ha the most expens electr in t...,[],[]
1,claim-375,when 3 per cent of total annual global emiss o...,"[evidence-116371, evidence-962087, evidence-11...",[this compar to 0.3 per cent per year in the p...
2,claim-1266,thi mean that the world is now 1c warmer than ...,[evidence-694262],[the planet is now warmer than in time]
3,claim-871,as it happen zika may also be a good model of ...,[],[]
4,claim-2164,greenland ha onli lost a tini fraction of it i...,"[evidence-52981, evidence-790637]",[if iceberg calv has happen as an averag green...
...,...,...,...,...
149,claim-2400,suddenli label co2 as a pollut is a disservic ...,[],[]
150,claim-204,after a natur orbit driven warm atmospher carb...,[],[]
151,claim-1426,mani of the world s coral reef are alreadi bar...,[],[]
152,claim-698,a recent studi led by lawrenc livermor nation ...,[],[]


In [181]:
train_claims_data = load_data('../data/train-claims.json')
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_cls': preprocess_text_keep_stopwords(claim_details['claim_text']),
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
    lambda x: [evidence_data[evidence_id] for evidence_id in x]
)

train_claims_df

,claim_id,claim,claim_text_cls,evidence,label,evidence_texts
0,claim-1937,scientif evid co2 pollut higher co2 concentr a...,not onli is there no scientif evid that co2 is...,"[evidence-442946, evidence-1194317, evidence-1...",DISPUTED,[at veri high concentr 100 time atmospher conc...
1,claim-126,el niño drove record high global temperatur su...,el niño drove record high in global temperatur...,"[evidence-338219, evidence-1127398]",REFUTES,[while climat chang can be due to natur forc o...
2,claim-2510,pdo switch cool phase,in 1946 pdo switch to a cool phase,"[evidence-530063, evidence-984887]",SUPPORTS,[there is evid of revers in the prevail polar ...
3,claim-2021,weather channel john coleman provid evid convi...,weather channel john coleman provid evid that ...,"[evidence-1177431, evidence-782448, evidence-5...",DISPUTED,[there is no convinc scientif evid that human ...
4,claim-2449,januari cap month period global temperatur dro...,januari 2008 cap a 12 month period of global t...,"[evidence-1010750, evidence-91661, evidence-72...",NOT_ENOUGH_INFO,"[with averag temperatur +8.1 47, the iranian p..."
...,...,...,...,...,...,...
1223,claim-1504,climat scientist say aspect case hurrican harv...,climat scientist say that aspect of the case o...,"[evidence-1055682, evidence-1047356, evidence-...",SUPPORTS,[it a fact climat chang made hurrican harvey m...
1224,claim-243,5th assess report ipcc estim human emiss proba...,in it 5th assess report in 2013 the ipcc estim...,[evidence-916755],SUPPORTS,[the scientif consensus as of 2013 updat as st...
1225,claim-2302,sinc mid global temperatur warm around degr ce...,sinc the mid 1970 global temperatur have been ...,"[evidence-403673, evidence-889933, evidence-11...",NOT_ENOUGH_INFO,"[global warm of 1.5, multipl independ produc i..."
1226,claim-502,abnorm temperatur spike februari earlier month...,but abnorm temperatur spike in februari and ea...,"[evidence-97375, evidence-562427, evidence-521...",NOT_ENOUGH_INFO,[a lower air temperatur of was record in 2010 ...


In [182]:
# combine claim text and evidence texts
X_train = train_claims_df['claim_text_cls'] + train_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_train = train_claims_df['label']

X_dev = dev_claims_df['claim_text_cls'] + dev_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))

count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(X_train)
X_dev_count = count_vectorizer.transform(X_dev)

In [183]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest 
rf_classifier = RandomForestClassifier()
rf_classifier.fit(X_train_count, y_train)
y_pred = rf_classifier.predict(X_dev_count)
dev_claims_df["label"] = y_pred
dev_claims_df['evidences'] = dev_claims_df['top5_evidence_id'].apply(
    lambda x: ["evidence-" + str(evidence_id) for evidence_id in x]
)
dev_claims_df.drop(columns=['evidence_texts', 'top5_evidence_id'], inplace=True)
dev_claims_df.rename(columns={"claim_text_raw": "claim_text", "label": "claim_label"}, inplace=True)
dev_claims_df.set_index('claim_id', inplace=True)
dev_claims_df

,claim_text_cls,claim_label,evidences
claim_id,,,
claim-752,south australia ha the most expens electr in t...,SUPPORTS,[]
claim-375,when 3 per cent of total annual global emiss o...,SUPPORTS,"[evidence-evidence-116371, evidence-evidence-9..."
claim-1266,thi mean that the world is now 1c warmer than ...,SUPPORTS,[evidence-evidence-694262]
claim-871,as it happen zika may also be a good model of ...,SUPPORTS,[]
claim-2164,greenland ha onli lost a tini fraction of it i...,SUPPORTS,"[evidence-evidence-52981, evidence-evidence-79..."
...,...,...,...
claim-2400,suddenli label co2 as a pollut is a disservic ...,SUPPORTS,[]
claim-204,after a natur orbit driven warm atmospher carb...,SUPPORTS,[]
claim-1426,mani of the world s coral reef are alreadi bar...,SUPPORTS,[]


In [184]:
# convert to json file
from json import loads
result = dev_claims_df.to_json(orient="index")
with open('../data/output/dev-output.json', 'w') as f:
    f.write(result)